# Machine Match Table Agent — Professional Edition
### A Multi-Agent System for Predictive-Maintenance Machine Matching
The behaviour visible to the user is identical —the internals are rebuilt as a small, explicit **multi-agent
system** instead of a handful of free-standing functions.

| |  Professional Edition  ||
|---|---|---|
| Structure  | 5 cooperating agent classes + orchestrator |
| Retrieval  | **Adaptive retrieval** (widens search when results are thin) |
| Quality control  | **Reflection Agent** critiques results and can trigger a retry |
| LLM robustness | **Retry mechanism** with backoff + logged attempts |
| Ranking | Raw distance **+ Confidence Score + Confidence Tier** |
| Observability  | Structured **Agent Logs** (timestamped, per-agent trace) |
| Continuity | **Session Memory** (remembers what worked, informs future runs) |

## Agent Architecture

```mermaid
flowchart LR
    U["User Input"] --> P["Planner Agent\n(perceive)"]
    P -- "needs_more_info" --> U
    P -- "validated query" --> R["Retrieval Agent\n(tool, adaptive)"]
    R --> RF["Reflection Agent\n(self-critique)"]
    RF -- "retry: widen top_k" --> R
    RF -- "accept" --> D["Decision Agent\n(confidence + ranking)"]
    D --> E["Explanation Agent\n(LLM, retry-wrapped)"]
    E --> T["Final Ranked Table"]
    MEM[("Session Memory")] -.-> P
    MEM -.-> R
    R -.-> MEM
```

## Agent Roles

| Agent | Responsibility | Maps to agent-loop stage |
|---|---|---|
| **Planner Agent** | Validates user input; decides whether `Laser_Intensity` is actually required | Perceive |
| **Retrieval Agent** | Filters + ranks candidates by normalized numeric distance; widens the search adaptively if too few results come back | Retrieve (tool) |
| **Reflection Agent** | Judges whether the retrieved set is good enough; decides if a retry is worthwhile | Self-critique |
| **Decision Agent** | Converts raw distance into a **Confidence Score** / **Confidence Tier** and produces the final ranking | Decide |
| **Explanation Agent** | The one LLM call in the pipeline — writes a structured risk insight per machine, wrapped in a retry mechanism | Reason (LLM) |

All five agents share a single `AgentLogger` (full audit trail) and a
`SessionMemory` (so repeated queries for the same machine type get
smarter over time).

**Note on "embeddings":** it ranks machines with a normalized numeric-distance
score across the query's fields. That scoring logic is preserved exactly
as-is here (see `RetrievalAgent.retrieve`)


## 1. Setup

In [ ]:
!pip install groq pandas tabulate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import getpass
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional

import pandas as pd
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    except Exception:
        GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass("Enter your Groq API key: ")

client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"  # unchanged from the original notebook

print("Groq client ready. Using model:", MODEL)

Groq client ready. Using model: llama-3.3-70b-versatile


## 2. Configuration

 `laser_machine_types`  controls the dynamic
slot-filling logic (`Laser_Intensity` is only required when
`Machine_Type` is one of these) — update it once you know the real
category values in your dataset. The new fields (`min_results`,
`max_top_k`, `llm_max_attempts`, ...) tune the Adaptive Retrieval,
Retry Mechanism, and Confidence Score behaviour introduced below.

In [ ]:
@dataclass
class AgentConfig:
    """Central, typed configuration for the Machine Match agentic pipeline."""

    # Planner
    laser_machine_types: set = field(default_factory=lambda: {"Laser_Cutter"})

    # Retrieval / Adaptive Retrieval
    default_top_k: int = 5
    min_results: int = 3
    max_top_k: int = 20
    top_k_step: int = 5

    # Reflection / Retry Mechanism (retrieval side)
    max_retrieval_retries: int = 2

    # Explanation Agent / Retry Mechanism (LLM side)
    llm_max_attempts: int = 3
    llm_backoff_seconds: float = 1.5
    llm_temperature: float = 0.5

    # Decision Agent / Confidence Score
    # Match_Score values at or beyond this are treated as ~0% confidence.
    confidence_scale: float = 3.0


CONFIG = AgentConfig()

## 3. Agent Logs

Every agent writes to one shared `AgentLogger` instead of calling
`print()`. The result is a structured, timestamped trace of exactly what
each agent did, in order — planning decisions, adaptive widening,
reflection verdicts, ranking, and every LLM retry attempt. It renders as
a plain DataFrame at the end of the run.

In [ ]:
@dataclass
class AgentLog:
    timestamp: str
    agent: str
    action: str
    detail: str
    status: str = "ok"


class AgentLogger:
    """Collects a structured, chronological trace of everything every agent does."""

    def __init__(self):
        self._logs: list[AgentLog] = []

    def log(self, agent: str, action: str, detail: str, status: str = "ok") -> None:
        self._logs.append(AgentLog(
            timestamp=datetime.now().strftime("%H:%M:%S"),
            agent=agent,
            action=action,
            detail=detail,
            status=status,
        ))

    def as_dataframe(self) -> pd.DataFrame:
        if not self._logs:
            return pd.DataFrame(columns=["timestamp", "agent", "action", "detail", "status"])
        return pd.DataFrame([log.__dict__ for log in self._logs])

    def clear(self) -> None:
        self._logs.clear()

    def __repr__(self) -> str:
        return f"AgentLogger({len(self._logs)} entries)"

## 4. Session Memory

A lightweight working memory shared across runs in the same session. It
does two things:

1. Keeps a full history of every query and result, so past runs stay
   inspectable (`memory.recent()`).
2. Remembers which `top_k` finally satisfied a given `Machine_Type`, so
   the next query for that same type starts **Adaptive Retrieval** from a
   smarter default instead of always starting cold.

In [ ]:
class SessionMemory:
    """Working memory for the pipeline: query history + adaptive top_k hints."""

    def __init__(self):
        self.history: list[dict] = []
        self._top_k_hints: dict[str, int] = {}

    def suggested_top_k(self, machine_type: str, default: int) -> int:
        """Returns the top_k that worked last time for this machine type, if any."""
        return self._top_k_hints.get(machine_type, default)

    def remember(self, machine_type: str, query: dict, result_count: int,
                 final_top_k: int, status: str) -> None:
        self.history.append({
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "machine_type": machine_type,
            "query": query,
            "result_count": result_count,
            "final_top_k": final_top_k,
            "status": status,
        })
        if result_count > 0:
            self._top_k_hints[machine_type] = final_top_k

    def recent(self, n: int = 5) -> pd.DataFrame:
        if not self.history:
            return pd.DataFrame(columns=["timestamp", "machine_type", "query", "result_count", "final_top_k", "status"])
        return pd.DataFrame(self.history[-n:])


MEMORY = SessionMemory()

## 5. Planner Agent — Perceive

Validates the user's specs and decides whether `Laser_Intensity` is
actually needed, rather than assuming every machine type requires it.
Same logic as the original `build_query`, now an agent with its own
logging.

In [ ]:
@dataclass
class PlanResult:
    status: str                       # "ok" | "needs_more_info"
    query: Optional[dict] = None
    missing_field: Optional[str] = None
    reason: Optional[str] = None


class PlannerAgent:
    """Perceives the user's request and turns it into a validated retrieval query."""

    def __init__(self, config: AgentConfig, logger: AgentLogger):
        self.config = config
        self.logger = logger

    def plan(self, machine_type: str, installation_year: int, maintenance_history_count: int,
             laser_intensity: Optional[float] = None) -> PlanResult:
        needs_laser = machine_type in self.config.laser_machine_types

        if needs_laser and laser_intensity is None:
            reason = f"{machine_type} typically reports Laser_Intensity; please provide it."
            self.logger.log("Planner", "validate", reason, status="needs_more_info")
            return PlanResult(status="needs_more_info", missing_field="Laser_Intensity", reason=reason)

        query = {
            "Machine_Type": machine_type,
            "Installation_Year": installation_year,
            "Maintenance_History_Count": maintenance_history_count,
        }
        if needs_laser:
            query["Laser_Intensity"] = laser_intensity

        self.logger.log("Planner", "build_query", f"query={query}")
        return PlanResult(status="ok", query=query)

## 6. Retrieval Agent — Retrieve (Tool) + Adaptive Retrieval

Filters candidates by exact `Machine_Type`, then ranks them by the same
normalized-distance score as the original notebook — pure pandas, no LLM
call. `adaptive_retrieve` is new: if the first pass returns fewer than
`min_results`, it widens `top_k` on its own (up to `max_top_k`) before
handing control to the Reflection Agent.

In [ ]:
class RetrievalAgent:
    """Retrieves and ranks candidate machines by normalized numeric distance."""

    def __init__(self, df: pd.DataFrame, config: AgentConfig, logger: AgentLogger):
        self.df = df
        self.config = config
        self.logger = logger

    def retrieve(self, query: dict, top_k: int) -> pd.DataFrame:
        candidates = self.df[self.df["Machine_Type"] == query["Machine_Type"]].copy()
        if candidates.empty:
            self.logger.log("Retrieval", "retrieve",
                             f"No rows for Machine_Type={query['Machine_Type']}", status="empty")
            return candidates

        numeric_fields = [f for f in ("Installation_Year", "Maintenance_History_Count", "Laser_Intensity")
                           if f in query]
        score = pd.Series(0.0, index=candidates.index)
        for field_name in numeric_fields:
            scale = self.df[field_name].std() or 1.0
            score += (candidates[field_name] - query[field_name]).abs() / scale

        candidates["Match_Score"] = score.round(3)
        ranked = candidates.sort_values("Match_Score").head(top_k)
        self.logger.log("Retrieval", "retrieve", f"top_k={top_k} -> {len(ranked)} candidates")
        return ranked

    def adaptive_retrieve(self, query: dict, start_top_k: int) -> tuple[pd.DataFrame, int]:
        """Widens top_k progressively until min_results is met or max_top_k is hit."""
        top_k = start_top_k
        matches = self.retrieve(query, top_k)
        while len(matches) < self.config.min_results and top_k < self.config.max_top_k:
            top_k = min(top_k + self.config.top_k_step, self.config.max_top_k)
            self.logger.log("Retrieval", "adaptive_widen",
                             f"only {len(matches)} results so far, widening to top_k={top_k}")
            matches = self.retrieve(query, top_k)
        return matches, top_k

## 7. Reflection Agent — Self-Critique

The self-critique step of the agent loop. It doesn't retrieve or reason
about content — it judges whether what came back from the Retrieval
Agent is good enough to hand to Decision and Explanation, and can request
one more retrieval pass (bounded by `max_retrieval_retries`).

In [ ]:
@dataclass
class ReflectionResult:
    should_retry: bool
    reason: str
    suggested_top_k: Optional[int] = None


class ReflectionAgent:
    """Reviews retrieval quality and decides whether the pipeline should retry."""

    def __init__(self, config: AgentConfig, logger: AgentLogger):
        self.config = config
        self.logger = logger

    def reflect(self, matches: pd.DataFrame, current_top_k: int, attempt: int) -> ReflectionResult:
        if matches.empty:
            result = ReflectionResult(False, "No candidates at all -- retrying will not help.")
        elif (len(matches) < self.config.min_results
              and current_top_k < self.config.max_top_k
              and attempt < self.config.max_retrieval_retries):
            suggested = min(current_top_k + self.config.top_k_step, self.config.max_top_k)
            result = ReflectionResult(
                True,
                f"Only {len(matches)} matches, below min_results={self.config.min_results}.",
                suggested,
            )
        else:
            result = ReflectionResult(False, f"{len(matches)} matches judged sufficient.")

        self.logger.log("Reflection", "review", result.reason,
                         status="retry" if result.should_retry else "accept")
        return result

## 8. Decision Agent — Confidence Score + Ranking

Converts the raw `Match_Score` (lower = closer match) into a
human-readable **Confidence Score** (0–100%) and a **Confidence Tier**
(`Low` / `Medium` / `High`), then produces the final ranking.

In [ ]:
class DecisionAgent:
    """Scores confidence and ranks the final shortlist from Match_Score."""

    def __init__(self, config: AgentConfig, logger: AgentLogger):
        self.config = config
        self.logger = logger

    def decide(self, matches: pd.DataFrame) -> pd.DataFrame:
        ranked = matches.sort_values("Match_Score").reset_index(drop=True).copy()

        confidence = (1 - (ranked["Match_Score"] / self.config.confidence_scale)).clip(lower=0, upper=1) * 100
        ranked["Confidence_Score"] = confidence.round(1)
        ranked["Confidence_Tier"] = pd.cut(
            ranked["Confidence_Score"], bins=[-1, 40, 70, 100], labels=["Low", "Medium", "High"]
        )

        self.logger.log(
            "Decision", "rank",
            f"Ranked {len(ranked)} machines; avg confidence={ranked['Confidence_Score'].mean():.1f}%"
        )
        return ranked

## 9. Explanation Agent — Reason (LLM, Structured JSON, Retry-Wrapped)

The one LLM agent in the pipeline — same system prompt, same Groq model,
same "return only a JSON array" contract as the original `annotate_matches`.
The new part is the **retry mechanism**: transient API errors and
unparsable output are retried up to `llm_max_attempts` times with a
linear backoff, and every attempt is written to the Agent Logs instead of
failing silently.

In [ ]:
class ExplanationAgent:
    """Writes one short, structured risk insight per matched machine via Groq."""

    SYSTEM_PROMPT = (
        "You are a Predictive Maintenance Risk Annotator. For each machine record given, "
        "write a single short risk insight (max 15 words) based ONLY on the fields provided "
        "(e.g. failure history, days since last maintenance, remaining useful life). "
        "Return ONLY a JSON array, one object per machine, in the same order as given, "
        "each with exactly two keys: \"Machine_ID\" and \"AI_Risk_Insight\". "
        "Do not include any text outside the JSON array, no markdown fences."
    )

    def __init__(self, client: Groq, model: str, config: AgentConfig, logger: AgentLogger):
        self.client = client
        self.model = model
        self.config = config
        self.logger = logger

    def explain(self, matches: pd.DataFrame) -> pd.DataFrame:
        if matches.empty or "Machine_ID" not in matches.columns:
            return matches

        records = matches.to_dict(orient="records")
        user_prompt = f"Machine records:\n{json.dumps(records, default=str)}"

        last_error: Optional[Exception] = None
        for attempt in range(1, self.config.llm_max_attempts + 1):
            try:
                response = self.client.chat.completions.create(
                    model=self.model,
                    temperature=self.config.llm_temperature,
                    messages=[
                        {"role": "system", "content": self.SYSTEM_PROMPT},
                        {"role": "user", "content": user_prompt},
                    ],
                )
                raw = response.choices[0].message.content.strip().strip("`")
                if raw[:4].lower() == "json":
                    raw = raw[4:].strip()

                annotations = json.loads(raw)
                ann_df = pd.DataFrame(annotations)
                merged = matches.merge(ann_df, on="Machine_ID", how="left")
                self.logger.log("Explanation", "annotate", f"attempt {attempt}: got {len(ann_df)} insights")
                return merged

            except Exception as exc:  # malformed JSON, API error, etc.
                last_error = exc
                self.logger.log("Explanation", "annotate", f"attempt {attempt} failed: {exc}", status="retry")
                if attempt < self.config.llm_max_attempts:
                    time.sleep(self.config.llm_backoff_seconds * attempt)

        fallback = matches.copy()
        fallback["AI_Risk_Insight"] = "Unavailable (agent output could not be parsed after retries)"
        self.logger.log("Explanation", "annotate",
                         f"All {self.config.llm_max_attempts} attempts failed: {last_error}", status="failed")
        return fallback

## 10. Orchestrator — Full Agent Loop

Wires the five agents together: **Planner -> Retrieval (adaptive) ->
Reflection (possible retry) -> Decision -> Explanation**, sharing one
`AgentLogger` and one `SessionMemory`, and returns a single clean table —
the matched machines, their confidence score/tier, and the AI-written
insight, with `Match_Score` kept last for reference.

In [ ]:
class MachineMatchOrchestrator:
    """Coordinates the Planner, Retrieval, Reflection, Decision, and
    Explanation agents into the full perceive -> retrieve -> reflect ->
    decide -> explain loop.
    """

    def __init__(self, df: pd.DataFrame, client: Groq, model: str,
                 config: Optional[AgentConfig] = None,
                 logger: Optional[AgentLogger] = None,
                 memory: Optional[SessionMemory] = None):
        self.config = config or AgentConfig()
        self.logger = logger or AgentLogger()
        self.memory = memory or SessionMemory()

        self.planner = PlannerAgent(self.config, self.logger)
        self.retrieval = RetrievalAgent(df, self.config, self.logger)
        self.reflection = ReflectionAgent(self.config, self.logger)
        self.decision = DecisionAgent(self.config, self.logger)
        self.explanation = ExplanationAgent(client, model, self.config, self.logger)

    def run(self, machine_type: str, installation_year: int, maintenance_history_count: int,
            laser_intensity: Optional[float] = None) -> dict:
        self.logger.clear()
        self.logger.log("Orchestrator", "start", f"machine_type={machine_type}")

        plan = self.planner.plan(machine_type, installation_year, maintenance_history_count, laser_intensity)
        if plan.status == "needs_more_info":
            return {"status": "needs_more_info", "reason": plan.reason, "logs": self.logger.as_dataframe()}

        start_top_k = self.memory.suggested_top_k(machine_type, self.config.default_top_k)
        matches, top_k = self.retrieval.adaptive_retrieve(plan.query, start_top_k)

        attempt = 0
        while True:
            reflection = self.reflection.reflect(matches, top_k, attempt)
            if not reflection.should_retry:
                break
            attempt += 1
            top_k = reflection.suggested_top_k
            matches = self.retrieval.retrieve(plan.query, top_k)

        if matches.empty:
            self.memory.remember(machine_type, plan.query, 0, top_k, "no_match")
            self.logger.log("Orchestrator", "finish", "No matching machines found.", status="empty")
            return {"status": "ok", "table": pd.DataFrame(), "message": "No matching machines found.",
                    "logs": self.logger.as_dataframe()}

        ranked = self.decision.decide(matches)
        annotated = self.explanation.explain(ranked)

        # Column order: raw fields -> AI_Risk_Insight -> Confidence_Score -> Confidence_Tier -> Match_Score
        tail = ["AI_Risk_Insight", "Confidence_Score", "Confidence_Tier", "Match_Score"]
        ordered_cols = [c for c in annotated.columns if c not in tail] + [c for c in tail if c in annotated.columns]
        table = annotated[ordered_cols].reset_index(drop=True)

        self.memory.remember(machine_type, plan.query, len(table), top_k, "ok")
        self.logger.log("Orchestrator", "finish", f"Returned {len(table)} ranked machines.")

        return {"status": "ok", "table": table, "logs": self.logger.as_dataframe()}

## 11. Load Dataset

Same dataset and download path as the original notebook.

In [ ]:
import kagglehub

path = kagglehub.dataset_download('canozensoy/industrial-iot-dataset-synthetic')
CSV_PATH = os.path.join(path, 'factory_sensor_simulator_2040.csv')
df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df):,} rows from {CSV_PATH}")
df.head()

100%|██████████| 16.1M/16.1M [00:00<00:00, 185MB/s]

Extracting files...


Loaded 500,000 rows from /root/.cache/kagglehub/datasets/canozensoy/industrial-iot-dataset-synthetic/versions/1/factory_sensor_simulator_2040.csv


,Machine_ID,Machine_Type,Installation_Year,Operational_Hours,Temperature_C,Vibration_mms,Sound_dB,Oil_Level_pct,Coolant_Level_pct,Power_Consumption_kW,...,Failure_History_Count,AI_Supervision,Error_Codes_Last_30_Days,Remaining_Useful_Life_days,Failure_Within_7_Days,Laser_Intensity,Hydraulic_Pressure_bar,Coolant_Flow_L_min,Heat_Index,AI_Override_Events
0,MC_000000,Mixer,2027,81769,73.43,12.78,83.72,36.76,68.74,84.95,...,5,True,3,162.0,False,NaN,NaN,NaN,NaN,2
1,MC_000001,Industrial_Chiller,2032,74966,58.32,14.99,77.04,100.00,62.13,154.61,...,2,True,4,147.0,False,NaN,NaN,40.92,NaN,2
2,MC_000002,Pick_and_Place,2003,94006,49.63,23.78,69.08,42.96,35.96,51.90,...,1,True,6,0.0,True,NaN,NaN,NaN,NaN,2
3,MC_000003,Vision_System,2007,76637,63.73,12.38,85.58,94.90,48.94,75.61,...,1,False,4,161.0,False,NaN,NaN,NaN,NaN,0
4,MC_000004,Shuttle_System,2016,20870,42.77,4.42,96.72,47.56,53.78,224.93,...,2,False,1,765.0,False,NaN,NaN,NaN,NaN,0


## 12. Demo Run

Enter the machine
specs, and `Laser_Intensity` is only asked for when it's actually needed.
Adaptive retrieval, reflection, confidence scoring,
retries, and logging — happens inside the orchestrator.

In [ ]:
orchestrator = MachineMatchOrchestrator(df, client, MODEL, config=CONFIG, memory=MEMORY)

# --- Collect specs from the user ---
machine_type = input("Machine Type: ").strip()
installation_year = int(input("Installation Year: ").strip())
maintenance_history_count = int(input("Maintenance History Count: ").strip())

# Only ask for Laser_Intensity if it's actually needed
laser_intensity = None
if machine_type in CONFIG.laser_machine_types:
    laser_intensity = float(input("Laser Intensity: ").strip())

result = orchestrator.run(
    machine_type=machine_type,
    installation_year=installation_year,
    maintenance_history_count=maintenance_history_count,
    laser_intensity=laser_intensity,
)

if result["status"] == "needs_more_info":
    print("Agent needs more info:", result["reason"])
elif result["table"].empty:
    print(result.get("message", "No matches found."))
else:
    display("found")

Machine Type: Mixer
Installation Year: 2027
Maintenance History Count: 4


'found'

### Agent Logs

The full, timestamped trace of what every agent did on this run — planning, adaptive widening, reflection verdicts, ranking, and every LLM retry attempt.

In [ ]:
display(result["logs"])

,timestamp,agent,action,detail,status
0,00:47:26,Orchestrator,start,machine_type=Mixer,ok
1,00:47:26,Planner,build_query,"query={'Machine_Type': 'Mixer', 'Installation_...",ok
2,00:47:26,Retrieval,retrieve,top_k=5 -> 5 candidates,ok
3,00:47:26,Reflection,review,5 matches judged sufficient.,accept
4,00:47:26,Decision,rank,Ranked 5 machines; avg confidence=100.0%,ok
5,00:47:27,Explanation,annotate,attempt 1: got 5 insights,ok
6,00:47:27,Orchestrator,finish,Returned 5 ranked machines.,ok


### Session Memory

What the pipeline remembers so far this session — useful to confirm Adaptive Retrieval is actually reusing a smarter `top_k` on repeat queries.

In [ ]:
display(orchestrator.memory.recent())

,timestamp,machine_type,query,result_count,final_top_k,status
0,00:47:27,Mixer,"{'Machine_Type': 'Mixer', 'Installation_Year':...",5,5,ok


### Markdown version (for copy-pasting into a report)

In [ ]:
if result["status"] == "ok" and not result["table"].empty:
    print(result["table"].to_markdown(index=False))

| Machine_ID   | Machine_Type   |   Installation_Year |   Operational_Hours |   Temperature_C |   Vibration_mms |   Sound_dB |   Oil_Level_pct |   Coolant_Level_pct |   Power_Consumption_kW |   Last_Maintenance_Days_Ago |   Maintenance_History_Count |   Failure_History_Count | AI_Supervision   |   Error_Codes_Last_30_Days |   Remaining_Useful_Life_days | Failure_Within_7_Days   |   Laser_Intensity |   Hydraulic_Pressure_bar |   Coolant_Flow_L_min |   Heat_Index |   AI_Override_Events | AI_Risk_Insight           |   Confidence_Score | Confidence_Tier   |   Match_Score |
|:-------------|:---------------|--------------------:|--------------------:|----------------:|----------------:|-----------:|----------------:|--------------------:|-----------------------:|----------------------------:|----------------------------:|------------------------:|:-----------------|---------------------------:|-----------------------------:|:------------------------|------------------:|----------------------